In [2]:
import pandas as pd

In [ ]:
# Peak first lines to validate data
with open("datadump_qie_csv.csv", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline())

"Number,Time (ms),Bus,Direction,Type,ID (hex),Reserved,Length,D0,D1,D2,D3,D4,D5,D6,D7,D...,"

"1,0.401,1,Rx,DT,0CFF2621,-,8,F3,F3,FF,FF,FF,FF,FF,FF"

"2,1.038,1,Rx,DT,0CF00321,-,8,FF,00,00,FF,FF,FF,FF,FF"

"3,1.670,1,Rx,DT,18FEF121,-,8,F7,FF,FF,FF,FF,FF,FF,FF"

"4,2.298,1,Rx,DT,0C010306,-,8,F3,FF,7D,FF,FF,FD,CB,3F"



In [20]:
# Load data
df_raw = pd.read_csv("datadump_qie_csv.csv", header=None)

# Split into multiple columns manually
df = df_raw[0].str.split(",", expand=True)

# Set first row to header and remove from data
df.columns = df.iloc[0]  # First row becomes header
df = df.drop(index=0)    # Remove the header row from data

# Drop completely empty columns
df = df.dropna(axis=1, how='all')

# Print for validation
df.head()

,Number,Time (ms),Bus,Direction,Type,ID (hex),Reserved,Length,D0,D1,D2,D3,D4,D5,D6,D7
1,1,0.401,1,Rx,DT,0CFF2621,-,8,F3,F3,FF,FF,FF,FF,FF,FF
2,2,1.038,1,Rx,DT,0CF00321,-,8,FF,00,00,FF,FF,FF,FF,FF
3,3,1.670,1,Rx,DT,18FEF121,-,8,F7,FF,FF,FF,FF,FF,FF,FF
4,4,2.298,1,Rx,DT,0C010306,-,8,F3,FF,7D,FF,FF,FD,CB,3F
5,5,5.321,1,Rx,DT,0CFF5003,-,8,FF,FF,FF,FF,FF,12,FF,FF


In [21]:
def parse_j1939_id(can_id_hex):
    # Convert hex string to integer
    can_id = int(can_id_hex, 16)

    # Extract fields
    priority = (can_id >> 26) & 0x7
    pgn_high = (can_id >> 16) & 0xFF  # PF
    pgn_low = (can_id >> 8) & 0xFF    # PS
    sa = can_id & 0xFF                # Source Address

    # PGN calculation
    if pgn_high >= 240:  # PDU2 format
        pgn = (pgn_high << 8) | pgn_low
    else:                # PDU1 format
        pgn = pgn_high << 8

    return priority, pgn, pgn_high, pgn_low, sa

# Apply to DataFrame
df[['Priority', 'PGN', 'PF', 'PS', 'SA']] = df['ID (hex)'].apply(
    lambda x: pd.Series(parse_j1939_id(x))
)

df.head()

,Number,Time (ms),Bus,Direction,Type,ID (hex),Reserved,Length,D0,D1,...,D3,D4,D5,D6,D7,Priority,PGN,PF,PS,SA
1,1,0.401,1,Rx,DT,0CFF2621,-,8,F3,F3,...,FF,FF,FF,FF,FF,3,65318,255,38,33
2,2,1.038,1,Rx,DT,0CF00321,-,8,FF,00,...,FF,FF,FF,FF,FF,3,61443,240,3,33
3,3,1.670,1,Rx,DT,18FEF121,-,8,F7,FF,...,FF,FF,FF,FF,FF,6,65265,254,241,33
4,4,2.298,1,Rx,DT,0C010306,-,8,F3,FF,...,FF,FF,FD,CB,3F,3,256,1,3,6
5,5,5.321,1,Rx,DT,0CFF5003,-,8,FF,FF,...,FF,FF,12,FF,FF,3,65360,255,80,3


In [24]:
# Extract only the ISO21815-2 PGNs
i21815_pgns = [int(x, 16) for x in ["FACC", "FACB", "F210", "F211"]]
print(i21815_pgns)

# Filter dataframe
df_filtered = df[df["PGN"].isin(i21815_pgns)].copy()

print(df_filtered.head())
print(f"Rows kept: {len(df_filtered)}")

[64204, 64203, 61968, 61969]
0   Number Time (ms) Bus Direction Type  ID (hex) Reserved Length  D0  D1  \
16      16    11.604   1        Rx   DT  14FACB05        -      8  00  00   
153    153   112.941   1        Rx   DT  14FACB05        -      8  00  00   
295    295   214.632   1        Rx   DT  14FACB05        -      8  00  00   
439    439   322.274   1        Rx   DT  14FACB05        -      8  00  00   
582    582   422.257   1        Rx   DT  14FACB05        -      8  00  00   

0    ...  D3  D4  D5  D6  D7 Priority    PGN   PF   PS  SA  
16   ...  AB  FF  FF  FF  05        5  64203  250  203   5  
153  ...  AB  FF  FF  FF  06        5  64203  250  203   5  
295  ...  AB  FF  FF  FF  07        5  64203  250  203   5  
439  ...  AB  FF  FF  FF  08        5  64203  250  203   5  
582  ...  AB  FF  FF  FF  09        5  64203  250  203   5  

[5 rows x 21 columns]
Rows kept: 268


In [26]:
pgn_map = {
    0xF210: "CxD→MachineStatus (PROPULSION)",
    0xF211: "CxD→MachineCommand (PROPULSION)",
    0xFACC: "Machine→CxDdata (PROPULSION)",
    0xFACB: "Machine→CxDreply"
}

In [27]:
# ------------------------------
# Lookup tables from ISO 21815-2
# ------------------------------

SUBSYSTEM_MAP = {
    0b000: "PROPULSION",
    0b111: "PROTOCOL",
    # other reserved...
}

PROPULSION_ENQ0 = {
    0b110: "GET_PROPULSION_REGISTER"
}
PROPULSION_ACT0 = {
    0b100: "LOAD_PROPULSION_SETPOINTS",
    0b110: "SET_PROPULSION_REGISTER"
}

PROTOCOL_ENQ7 = {
    0b000: "PROTOCOL_NOP",
    0b001: "NEGOTIATE_NOP",
    0b010: "NEGOTIATE_ENQ"
}
PROTOCOL_ACT7 = {
    0b101: "RESET_REGISTERS",
    0b110: "SET_PROTOCOL_REGISTER"
}

# ------------------------------
# Decode helpers
# ------------------------------
def decode_status_byte(status_byte):
    """Decode SPN:Status field into subsystem + action/enquiry."""
    subsystem = (status_byte >> 4) & 0x07
    code = status_byte & 0x0F
    name = SUBSYSTEM_MAP.get(subsystem, f"Unknown({subsystem})")
    return name, code

def decode_f210(data_bytes):
    """Decode PGN F210 (CxD→MachineStatus)."""
    subsystem, code = decode_status_byte(data_bytes[0])
    reg_index = data_bytes[1]
    reg_select = data_bytes[2]
    value = int.from_bytes(data_bytes[3:7], byteorder='little')
    msg_id = data_bytes[7]

    # Action/Enquiry lookup
    if subsystem == "PROPULSION":
        action_name = PROPULSION_ACT0.get(code, f"UnknownCode({code})")
    elif subsystem == "PROTOCOL":
        action_name = PROTOCOL_ENQ7.get(code, f"UnknownCode({code})")
    else:
        action_name = f"UnknownSubsystemCode({code})"

    return {
        "Subsystem": subsystem,
        "Action/Enquiry": action_name,
        "RegisterIndex": reg_index,
        "RegisterSelect": reg_select,
        "Value": value,
        "MessageID": msg_id
    }

def decode_f211(data_bytes):
    """Decode PGN F211 (CxD→MachineCommand) — similar to F210 but for execution commands."""
    # Would use ISO table for MachineCommand bitfields
    return {
        "CommandRaw": data_bytes
    }

def decode_facc(data_bytes):
    """Decode PGN FACC (Machine→CxDdata, PROPULSION telemetry)."""
    speed = int.from_bytes(data_bytes[0:2], byteorder='little') * 0.0036  # m/s → km/h
    gear = data_bytes[2]
    direction = data_bytes[3]
    return {
        "Speed_kmh": round(speed, 2),
        "Gear": gear,
        "Direction": direction
    }

def decode_facb(data_bytes):
    """Decode PGN FACB (Machine→CxDreply)."""
    status = data_bytes[0]
    reg_index = data_bytes[1]
    return {
        "ReplyStatus": status,
        "RegisterIndex": reg_index
    }

# ------------------------------
# Dispatch table for PGNs
# ------------------------------
PGN_DECODERS = {
    0xF210: decode_f210,
    0xF211: decode_f211,
    0xFACC: decode_facc,
    0xFACB: decode_facb
}

def decode_message(pgn, data_bytes):
    """Call the correct decoder for a PGN if available."""
    decoder = PGN_DECODERS.get(pgn)
    if decoder:
        return decoder(data_bytes)
    else:
        return {"RawData": data_bytes}

In [31]:
# Example: decode the first row
row = df_filtered.iloc[0]

byte_cols = ['D0', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7']
row_bytes = [int(str(row[col]), 16) if pd.notna(row[col]) else 0 for col in byte_cols]

decoded = decode_message(row['PGN'], row_bytes)
print(decoded)

{'ReplyStatus': 0, 'RegisterIndex': 0}


In [ ]:
byte_cols = ['D0', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7']

df_filtered['Decoded'] = df_filtered.apply(
    lambda r: decode_message(
        r['PGN'],
        [int(str(r[c]), 16) if pd.notna(r[c]) else 0 for c in byte_cols]
    ),
    axis=1
)